# 1. Objective
  The Objective of the notebook is to process the future forecasts and generate the reports at Studio Group level and Region level
   - The border line weeks in the future month are adjusted to give forecasts for the exact number of days in the future month
   - Aggregate the SG level numbers to Region Level numbers using appropriate techniques

# 2. Imports

In [ ]:
# Import python packages
import streamlit as st
import pandas as pd
from snowflake.snowpark import functions as F
from functools import reduce
from operator import add

# We can also use Snowpark for our analyses!
from snowflake.snowpark.context import get_active_session
session = get_active_session()

# 3. Setup environment

## 3.1. Load Config

In [ ]:
import yaml
with open("config_new_PROD.yaml") as file:
    app_config = yaml.safe_load(file)

## 3.2. Update Output Database, Schema , table

In [ ]:
output_database = app_config["general_inputs"]["output_database"]
output_schema = app_config["general_inputs"]["output_schema"]
print(output_database, output_schema)

In [ ]:
session.use_database(output_database)
session.use_schema(output_schema)
original_output_table_name = "PROD_FINAL_MODEL_LASSO_COEFFICIENTS"
adjusted_output_table_name_sg_level = "PROD_BASELINE_REPORT_SG_LEVEL"
adjusted_output_table_name_region_level = "PROD_BASELINE_REPORT_REGION_LEVEL"

In [ ]:
# Example check (optional)
print("✅ Snowpark Session Initialized Successfully!")
print("Current Database:", session.get_current_database())
print("Current Schema:", session.get_current_schema())

# 4. Post Processing

## 4.1. Read Contributions data

In [ ]:
cont_df = session.table("PROD_TABLES.PROD_FINAL_MODEL_LASSO_CONTRIBUTIONS")

In [ ]:
#from snowflake.snowpark.functions import col, substr, lit, split
cont_df = cont_df.with_column(
    "REGIONNAME",
    F.substr(F.col("ITERATION_ID"), 1, 5)
)
cont_df = cont_df.with_column(
    "F_CODE",
    F.substr(F.col("ITERATION_ID"), 7, 4)
)

In [ ]:
cont_df

## 4.2. Generating Adjusted Forecasts
 For the future month, the first week and last week forecasts are adjusted so that it accounts for the exact number of days that falls in the immediate future month, ignoring the spillovers in previous and subsequent month

In [ ]:
day_weights_df = session.table("PROD_DAY_IMPORTANCE")

In [ ]:
day_weights_df = day_weights_df.with_column("DAY_OF_WEEK",F.col("DAY_OF_WEEK").cast("int"))
day_weights_df = day_weights_df.with_column("WEIGHTS",F.col("WEIGHTS").cast("float"))

In [ ]:
day_weights_df

In [ ]:
DAY_WEIGHTS = {
    int(row["DAY_OF_WEEK"]): row["WEIGHTS"]
    for row in day_weights_df.select("DAY_OF_WEEK", "WEIGHTS").collect()
}
DAY_WEIGHTS

In [ ]:
s=0
for i in DAY_WEIGHTS.values():
    s = s + i
print(s)

In [ ]:
# from snowflake.snowpark.functions import *
# from snowflake.snowpark import Session

# DAY_WEIGHTS = {
#     0: 0.15,  # Monday
#     1: 0.14,
#     2: 0.14,
#     3: 0.14,
#     4: 0.15,
#     5: 0.14,
#     6: 0.14   # Sunday
# }
# DAY_WEIGHTS

In [ ]:
def adjust_weekly_forecast_for_month(df, cols_to_adj,year, month):
    """
    Adjust only first & last week of the month using daily weights
    """

    # Month boundaries
    month_start = F.to_date(F.lit(f"{year}-{month:02d}-01"))
    month_end = F.last_day(month_start)

    # Expand week into 7 daily rows
    df_daily = (
        df
        .with_column("DAY_INDEX", F.explode(F.array_construct(F.lit(0),F.lit(1),F.lit(2),F.lit(3),F.lit(4),F.lit(5),F.lit(6))))
        .with_column("DAY_DATE", F.dateadd("day", F.col("DAY_INDEX"), F.col("DS")))
    )

    # Add daily weights
    df_daily = df_daily.with_column(
        "DAY_WEIGHT",
        F.when(F.col("DAY_INDEX") == 0, F.lit(DAY_WEIGHTS[0]))
        .when(F.col("DAY_INDEX") == 1, F.lit(DAY_WEIGHTS[1]))
        .when(F.col("DAY_INDEX") == 2, F.lit(DAY_WEIGHTS[2]))
        .when(F.col("DAY_INDEX") == 3, F.lit(DAY_WEIGHTS[3]))
        .when(F.col("DAY_INDEX") == 4, F.lit(DAY_WEIGHTS[4]))
        .when(F.col("DAY_INDEX") == 5, F.lit(DAY_WEIGHTS[5]))
        .otherwise(F.lit(DAY_WEIGHTS[6]))
    )

    # Keep only days that fall in the target month
    df_month_days = df_daily.filter(
        (F.col("DAY_DATE") >= month_start) &
        (F.col("DAY_DATE") <= month_end)
    )

    # Aggregate back to weekly (partial weeks automatically scaled)
    df_adjusted = (
        df_month_days
        .group_by("REGIONNAME","F_CODE","DS")
        .agg(
            #F.sum(col("PREDICTED_SALES") * col("DAY_WEIGHT")).alias("ADJUSTED_FORECAST"),
            #F.mean(col("PREDICTED_SALES")).alias("ORIG_FORECAST"),
            *[
            F.sum(F.col(str(c))* F.col("DAY_WEIGHT")).alias(str(c))   
            for c in cols_to_adj
        ],
        )
    )

    return df_adjusted

In [ ]:
cols_to_adj = [c for c in cont_df.columns if "CONTRIBUTION_" in c] + ["FINAL_BASE","FINAL_INCREMENTAL","PREDICTED_SALES","YHAT_UPPER_80","YHAT_LOWER_80"]

In [ ]:
cols_to_adj

In [ ]:
cont_df = cont_df.na.fill(0, cols_to_adj)

In [ ]:
cont_df = cont_df.with_column("DS",F.to_date(F.col("DS")))

In [ ]:
month_year = cont_df.filter(F.col("FUTURE_FLAG") == 0).select("DS").agg(F.max(F.col("DS")).alias("LAST_TRAINING_DATE")).with_column(
    "NEXT_MONTH",
    F.to_varchar(F.add_months(F.col("LAST_TRAINING_DATE"), 1), "MM-YYYY")
).to_pandas()["NEXT_MONTH"][0]
month, year = int(month_year.split("-")[0]),int(month_year.split("-")[1])
print(month, year)

In [ ]:
adjusted_df = adjust_weekly_forecast_for_month(
    cont_df,
    cols_to_adj = cols_to_adj,
    year=year,
    month=month
)

In [ ]:
adjusted_df

In [ ]:
hist_df = cont_df.select(["REGIONNAME","F_CODE","DS","FUTURE_FLAG"]+cols_to_adj).filter(F.col("FUTURE_FLAG") == 0)
future_df = adjusted_df
future_df = future_df.with_column("FUTURE_FLAG", F.lit(1))
final_adj_cont_df = hist_df.union_by_name(future_df)
print(final_adj_cont_df.count())

In [ ]:
holiday_cols = [c for c in cont_df.columns if (c.endswith("_IMPACT")) & ("DAY" in c)]

In [ ]:

from snowflake.snowpark.functions import col

# Use reduce to sum the columns iteratively
sum_expression = reduce(add, [F.col(c) for c in holiday_cols])

# Add the new column to the DataFrame
final_adj_cont_df = final_adj_cont_df.with_column("HOLIDAY_CONTRIBUTION", sum_expression)

# Display the result
final_adj_cont_df

In [ ]:
final_adj_cont_df = final_adj_cont_df.with_column("FINAL_BASE", F.col("FINAL_BASE") + F.col("HOLIDAY_CONTRIBUTION"))

In [ ]:
final_adj_cont_df

# QC

## Check if the borderline weeks are adjusted

In [ ]:
orig_data = cont_df.filter(F.col("FUTURE_FLAG") == 1)
orig_data = orig_data.with_column_renamed("PREDICTED_SALES","ORIGINAL_FORECAST")
adj_data = final_adj_cont_df.filter(F.col("FUTURE_FLAG") == 1)
adj_data = adj_data.with_column_renamed("PREDICTED_SALES","ADJUSTED_FORECAST")

comb_data = orig_data.join(adj_data.select(["REGIONNAME","F_CODE","DS","ADJUSTED_FORECAST"]), on = ["REGIONNAME","F_CODE","DS"], how = "left")
comb_data = comb_data.with_column("FORECAST_DIFF", F.abs(F.col("ADJUSTED_FORECAST") - F.col("ORIGINAL_FORECAST")))
comb_data = comb_data.with_column("FORECAST_CHECK",F.when(F.col("FORECAST_DIFF")>0.0001,1).otherwise(0))
agg_data = comb_data.group_by(["REGIONNAME","F_CODE"]).agg(
    F.sum(F.col("FORECAST_CHECK")).alias("WEEKS_ADJUSTED")
)
agg_data = agg_data.with_column("CORRECT_ADJUSTMENT", F.when(F.col("WEEKS_ADJUSTED") == 2, 1).otherwise(0))

rows_with_issues = agg_data.filter(F.col("CORRECT_ADJUSTMENT")==0).count()
if rows_with_issues>0:
    print("There are ",rows_with_issues," Studio groups with less than 2 weeks adjusted")
else:
    print("All good. 2 borderline weeks are adjusted for each REGION x F_CODE combination")
final_adj_cont_df = final_adj_cont_df.join(agg_data, ["REGIONNAME","F_CODE"], how = "left")
agg_data

In [ ]:
comb_data

## Check if Base and Incremental add up to Predicted sales / Forecasts

In [ ]:
final_adj_cont_df = final_adj_cont_df.with_column("PREDICTED_SALES_CALC", F.col("FINAL_BASE") + F.col("FINAL_INCREMENTAL"))
final_adj_cont_df = final_adj_cont_df.with_column("PERC_DIFF", (F.col("PREDICTED_SALES") - F.col("PREDICTED_SALES_CALC"))/F.col("PREDICTED_SALES"))
final_adj_cont_df = final_adj_cont_df.with_column("SIGNIFITCANT_DIFF", F.when(F.col("PERC_DIFF")>0.0001,1).otherwise(0))
rows_with_issues = final_adj_cont_df.filter(F.col("SIGNIFITCANT_DIFF")==1).count()
if rows_with_issues>0:
    print("There are ",rows_with_issues," rows with mismatch")
else:
    print("All good. Base and Incremental add up to Predicted Sales / Forecasts")

## Check if Adjusted Forecast is within the CI bounds

In [ ]:
final_adj_cont_df = final_adj_cont_df.with_column("WITHIN_BOUNDS",F.when((F.col("PREDICTED_SALES")>F.col("YHAT_LOWER_80")) & (F.col("PREDICTED_SALES")<F.col("YHAT_UPPER_80")),1).otherwise(0))

rows_with_issues = final_adj_cont_df.filter(F.col("WITHIN_BOUNDS")==0).count()
avg_forecast = final_adj_cont_df.filter(F.col("WITHIN_BOUNDS")==0).agg(F.mean(F.col("PREDICTED_SALES")).alias("AVERAGE_FORECAST")).collect()

if rows_with_issues>0:
    print("There are ",rows_with_issues," rows where adjusted forecast exceeds bounds and avg forecast is ->",avg_forecast)
else:
    print("All forecasts lie with in CI range")

final_adj_cont_df = final_adj_cont_df.with_column("YHAT_LOWER_80",F.when(F.col("PREDICTED_SALES")<F.col("YHAT_LOWER_80"),F.col("PREDICTED_SALES")+1).otherwise(F.col("YHAT_LOWER_80")))
final_adj_cont_df = final_adj_cont_df.with_column("YHAT_UPPER_80",F.when(F.col("PREDICTED_SALES")>F.col("YHAT_UPPER_80"),F.col("PREDICTED_SALES")+1).otherwise(F.col("YHAT_UPPER_80")))
#final_adj_cont_df.filter(F.col("WITHIN_BOUNDS")==0).agg(F.mean(F.col("PREDICTED_SALES")).alias("AVERAGE_FORECAST"))
final_adj_cont_df = final_adj_cont_df.with_column("WITHIN_BOUNDS_ADJ",F.when((F.col("PREDICTED_SALES")>F.col("YHAT_LOWER_80")) & (F.col("PREDICTED_SALES")<F.col("YHAT_UPPER_80")),1).otherwise(0))

rows_with_issues = final_adj_cont_df.filter(F.col("WITHIN_BOUNDS_ADJ")==0).count()

print("After adjustment, There are ",rows_with_issues," rows where adjusted forecast exceeds bounds")



In [ ]:
final_adj_cont_df.filter(F.col("WITHIN_BOUNDS")==0)

## Adjust Lower Bound if negative due to NCS calibration

In [ ]:
final_adj_cont_df = final_adj_cont_df.with_column("NEGATIVE_YHAT_LOWER_80",F.when(F.col("YHAT_LOWER_80")<0,1).otherwise(0))

rows_with_issues = final_adj_cont_df.filter(F.col("NEGATIVE_YHAT_LOWER_80")==1).count()
if rows_with_issues>0:
    print("There are ",rows_with_issues," rows where YHAT_LOWER is negative")
else:
    print("YHAT_LOWER is positive everywhere")

# Make negative values to 0
final_adj_cont_df = final_adj_cont_df.with_column("YHAT_LOWER_80",F.when(F.col("YHAT_LOWER_80")<0,0).otherwise(F.col("YHAT_LOWER_80")))
print("After adjustment, there are ",final_adj_cont_df.filter(F.col("YHAT_LOWER_80")<0).count(), " negative values")

## Check if there are any duplicate date entries

In [ ]:
agg_data2 = final_adj_cont_df.group_by(["REGIONNAME","F_CODE"]).agg(F.count_distinct(F.col("DS")).alias("UNIQUE_WEEKS"),
                                                       F.count(F.col("DS")).alias("AVAILABLE_WEEKS"))
agg_data2 = agg_data2.with_column("DUPLICATE_WEEKS", F.when(F.abs(F.col("UNIQUE_WEEKS") - F.col("AVAILABLE_WEEKS"))>0,0).otherwise(1))

rows_with_issues = agg_data2.filter(F.col("DUPLICATE_WEEKS")==0).count()
if rows_with_issues>0:
    print("There are ",rows_with_issues," SGs where there are duplicate weeks present")
else:
    print("No duplicate weeks found!")
    
agg_data2

## Check if base variables add up to final base

In [ ]:
base_cols = [c for c in final_adj_cont_df.columns if "_CONTRIBUTION_BASE" in c] + ["HOLIDAY_CONTRIBUTION"]
# Use reduce to sum the columns iteratively
sum_expression = reduce(add, [F.col(c) for c in base_cols])

# Add the new column to the DataFrame
final_adj_cont_df = final_adj_cont_df.with_column("FINAL_BASE_CALC", sum_expression)
final_adj_cont_df = final_adj_cont_df.with_column("FINAL_BASE_PERC_DIFF", (F.col("FINAL_BASE") - F.col("FINAL_BASE_CALC"))/F.col("FINAL_BASE"))
final_adj_cont_df = final_adj_cont_df.with_column("FINAL_BASE_SIGNIFITCANT_DIFF", F.when(F.col("FINAL_BASE_PERC_DIFF")>0.0001,1).otherwise(0))
rows_with_issues = final_adj_cont_df.filter(F.col("FINAL_BASE_SIGNIFITCANT_DIFF")==1).count()
if rows_with_issues>0:
    print("There are ",rows_with_issues," rows with mismatch")
else:
    print("All good. Base variables add up to Final Base")

## Check if impact variables add up to final incremental

In [ ]:
inc_cols = [c for c in final_adj_cont_df.columns if ("_CONTRIBUTION_IMPACT" in c) & ("DAY_" not in c)]
# Use reduce to sum the columns iteratively
sum_expression = reduce(add, [F.col(c) for c in inc_cols])

# Add the new column to the DataFrame
final_adj_cont_df = final_adj_cont_df.with_column("FINAL_INCREMENTAL_CALC", sum_expression)
final_adj_cont_df = final_adj_cont_df.with_column("FINAL_INCREMENTAL_PERC_DIFF", (F.col("FINAL_INCREMENTAL") - F.col("FINAL_INCREMENTAL_CALC")))
final_adj_cont_df = final_adj_cont_df.with_column("FINAL_INCREMENTAL_SIGNIFITCANT_DIFF", F.when(F.col("FINAL_INCREMENTAL_PERC_DIFF")>0.0001,1).otherwise(0))
rows_with_issues = final_adj_cont_df.filter(F.col("FINAL_INCREMENTAL_SIGNIFITCANT_DIFF")==1).count()
if rows_with_issues>0:
    print("There are ",rows_with_issues," rows with mismatch")
else:
    print("All good. Incremental variables add up to Final Incremental")

## 4.3. Baseline Report Generation

### 4.3.1. Studio Group level

In [ ]:
cols_to_agg_sg = [c for c in final_adj_cont_df.columns if c not in ["F_CODE","REGIONNAME","LOAD_TS","DS","ITERATION_ID","YEAR","LIFT","FUTURE_FLAG","PREDICTED_SALES_CALC","PERC_DIFF"]]

In [ ]:
cols_to_agg_sg

In [ ]:
studio_group_data = final_adj_cont_df.group_by(["REGIONNAME","F_CODE","DS"]).agg(
    *[
        F.sum(F.col(str(c))).alias(str(c))   
        for c in cols_to_agg_sg
    ]
)
studio_group_data

### 4.3.2. Region level

In [ ]:
cols_to_agg_region = [c for c in final_adj_cont_df.columns if c not in ["F_CODE","REGIONNAME","LOAD_TS","DS","ITERATION_ID","YEAR","LIFT","FUTURE_FLAG","PREDICTED_SALES_CALC","PERC_DIFF"]]

In [ ]:
region_data = final_adj_cont_df.group_by(["REGIONNAME","DS"]).agg(
    *[
        F.sum(F.col(str(c))).alias(str(c))   
        for c in cols_to_agg_region
    ]
)
region_data

# 5. Export reports

In [ ]:
# # write_file(pos, algo_path, "/Pos_standardization_results (")
#output_table_name = "REGION_LEVEL_CONTRIBUTIONS"
results_s_with_ts = studio_group_data.with_column("LOAD_TS", F.current_timestamp())
results_s_with_ts.write.mode("overwrite").save_as_table(adjusted_output_table_name_sg_level)
print(results_s_with_ts.count())

In [ ]:
# # write_file(pos, algo_path, "/Pos_standardization_results (")
#output_table_name = "REGION_LEVEL_CONTRIBUTIONS"
results_s_with_ts = region_data.with_column("LOAD_TS", F.current_timestamp())
results_s_with_ts.write.mode("overwrite").save_as_table(adjusted_output_table_name_region_level)
print(results_s_with_ts.count())